<a href="https://colab.research.google.com/github/Ophelie4/pfizer-document-intelligence/blob/main/project-02-python-for-document-processing/notebooks/Ophelie_Text_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Text Cleaning & Standardization

Cleans `text_sample.txt` (typos, accented/encoding issues, emojis, HTML tags, inconsistent price formats, uncontracted words, extra punctuation/spaces) and saves the result to `cleaned_text.txt`.

**Setup:** install the two cleaning libraries and write the sample file into the Colab session.

In [ ]:
!pip install -q ftfy contractions

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.4 MB/s eta 0:00:00


In [ ]:
%%writefile text_sample.txt
Thís ís à próblemátic téxt fíle!! It contains **extra spaces** ,,,,, special characters!!!💥🔥🚀
Some words are misspelleddd, and encoding issues liké thís cäusé problëms.

Prices are inconsistent: $29.99, 29.99 USD, 29,99$.

Emails & phone numbers may be embedded: contact@domain.com, (123)-456-7890.

Repeated punctuations!!!!! should be removed, along with **random symbols** like @@,##.

stopwords like "the", "is", and "a" appear often.

HTML tags might be present: <div>This is inside a div</div>

And sometimes, contractions won't expand: "can't", "won't", "shouldn't".

Random numeric values: 123456, 98765, 2024.


Writing text_sample.txt


## Read the raw file

In [ ]:
with open('text_sample.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(text)

Thís ís à próblemátic téxt fíle!! It contains **extra spaces** ,,,,, special characters!!!💥🔥🚀
Some words are misspelleddd, and encoding issues liké thís cäusé problëms.

Prices are inconsistent: $29.99, 29.99 USD, 29,99$.

Emails & phone numbers may be embedded: contact@domain.com, (123)-456-7890.

Repeated punctuations!!!!! should be removed, along with **random symbols** like @@,##.

stopwords like "the", "is", and "a" appear often.

HTML tags might be present: <div>This is inside a div</div>

And sometimes, contractions won't expand: "can't", "won't", "shouldn't".

Random numeric values: 123456, 98765, 2024.



## Clean the text

Steps, in order (order matters — see notes inline):
1. Fix encoding/mojibake issues with `ftfy`
2. Strip HTML tags
3. Expand contractions (`won't` → `will not`) while apostrophes still exist
4. Normalize accented characters to plain ASCII and drop emojis in the same pass — `unicodedata.normalize('NFKD', ...)` decomposes a letter like `í` into `i` + a separate accent mark, then `.encode('ascii', 'ignore')` drops the accent mark (and any emoji, which has no ASCII form) while keeping the base letter
5. Lowercase everything
6. Remove random symbol runs like `@@`, `##`
7. Standardize inconsistent price formats to `$X.XX`
8. Remove remaining punctuation (keeping `$` and `.` so prices stay readable)
9. Collapse repeated spaces/newlines into single spaces

In [ ]:
import re, string, unicodedata
import ftfy
import contractions

# 1. Fix encoding issues (mojibake)
text = ftfy.fix_text(text)

# 2. Remove HTML tags
text = re.sub(r'<[^>]+>', ' ', text)

# 3. Expand contractions (before stripping punctuation, needs the apostrophes)
text = contractions.fix(text)

# 4. Normalize accented characters to plain ASCII AND strip emojis in one pass
text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('ascii')

# 5. Lowercase
text = text.lower()

# 6. Remove random symbol runs like @@, ##
text = re.sub(r'[@#]{2,}', ' ', text)

# 7. Standardize price formats to $X.XX
text = re.sub(r'(\d+)[.,](\d{2})\s*usd', r'$\1.\2', text)
text = re.sub(r'(\d+)[.,](\d{2})\$', r'$\1.\2', text)

# 8. Remove remaining punctuation (keep $ and . so prices stay readable)
keep = string.punctuation.replace('$', '').replace('.', '')
text = text.translate(str.maketrans('', '', keep))

# 9. Collapse extra spaces/newlines
text = re.sub(r'\s+', ' ', text).strip()

print(text)

this is a problematic text file it contains extra spaces special characters some words are misspelleddd and encoding issues like this cause problems. prices are inconsistent $29.99 $29.99 $29.99. emails phone numbers may be embedded contactdomain.com 1234567890. repeated punctuations should be removed along with random symbols like . stopwords like the is and a appear often. html tags might be present this is inside a div and sometimes contractions will not expand cannot will not should not. random numeric values 123456 98765 2024.


## Save the cleaned text

In [ ]:
with open('cleaned_text.txt', 'w', encoding='utf-8') as f:
    f.write(text)

print('Saved cleaned_text.txt')

Saved cleaned_text.txt
